In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
import lightgbm as lgb
from sklearn.metrics import average_precision_score, roc_auc_score, precision_recall_curve, classification_report
from sklearn.isotonic import IsotonicRegression
from sklearn.model_selection import train_test_split
import shap
import joblib

DATA_DIR = Path.home() / "kkbox-churn" / "data" / "parquet"

master = pd.read_parquet(DATA_DIR / "master_dataset.parquet")

drop_cols = [
    'msno', 'last_transaction_date', 'last_expire_date',
    'last_listen_date', 'first_listen_date',
    'cluster_k5', 'segment',
    'churn_prob', 'monthly_revenue', 'churn_prob_clipped',
    'clv', 'retention_priority'
]
master = master.drop(columns=drop_cols)

for col in ['city', 'registered_via', 'bd_bucket', 'auto_renew_switch']:
    master[col] = master[col].astype('category')

master_raw = pd.read_parquet(DATA_DIR / "master_dataset.parquet")
apr_idx = master_raw[master_raw['last_expire_date'].dt.to_period('M') == '2017-04'].index
may_idx = master_raw[master_raw['last_expire_date'].dt.to_period('M') == '2017-05'].index

X = master.drop(columns=['is_churn'])
y = master['is_churn']

X_train, y_train = X.loc[apr_idx], y.loc[apr_idx]
X_val,   y_val   = X.loc[may_idx], y.loc[may_idx]

print(f"Train: {X_train.shape}, churn rate: {y_train.mean():.4f}")
print(f"Val:   {X_val.shape},   churn rate: {y_val.mean():.4f}")

Train: (800236, 101), churn rate: 0.0165
Val:   (79942, 101),   churn rate: 0.0502


In [3]:
# Cell 2 — Train final model
best_params = {
    'objective': 'binary',
    'metric': 'auc',
    'boosting_type': 'gbdt',
    'n_estimators': 2000,
    'learning_rate': 0.05386267542952244,
    'num_leaves': 39,
    'min_child_samples': 159,
    'feature_fraction': 0.8526646900411762,
    'bagging_fraction': 0.8828494997505784,
    'bagging_freq': 1,
    'reg_alpha': 0.10229096094071723,
    'reg_lambda': 1.1827367316767054e-05,
    'min_split_gain': 0.7253877757449226,
    'is_unbalance': True,
    'random_state': 42,
    'n_jobs': -1,
    'verbose': -1
}

final_model = lgb.LGBMClassifier(**best_params)
final_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    callbacks=[
        lgb.early_stopping(stopping_rounds=50, verbose=True),
        lgb.log_evaluation(period=50)
    ]
)

val_preds = final_model.predict_proba(X_val)[:, 1]
roc = roc_auc_score(y_val, val_preds)
pr = average_precision_score(y_val, val_preds)
print(f"\nROC-AUC: {roc:.4f}")
print(f"PR-AUC:  {pr:.4f}")
print(f"Best iteration: {final_model.best_iteration_}")

model_path = Path.home() / "kkbox-churn" / "models" / "lgbm_churn_final.pkl"
model_path.parent.mkdir(exist_ok=True)
joblib.dump(final_model, model_path)
print(f"Saved: {model_path}")

Training until validation scores don't improve for 50 rounds
[50]	valid_0's auc: 0.934263
[100]	valid_0's auc: 0.944021
[150]	valid_0's auc: 0.94581
[200]	valid_0's auc: 0.947753
Early stopping, best iteration is:
[185]	valid_0's auc: 0.948066

ROC-AUC: 0.9481
PR-AUC:  0.4287
Best iteration: 185
Saved: /Users/harshithnr/kkbox-churn/models/lgbm_churn_final.pkl


In [4]:
val_preds_full = final_model.predict_proba(X_val)[:, 1]

pr_auc = average_precision_score(y_val, val_preds_full)
print(f"ROC-AUC:  {roc_auc_score(y_val, val_preds_full):.4f}")
print(f"PR-AUC:   {pr_auc:.4f}")

# At 0.5 threshold
preds_binary = (val_preds_full >= 0.5).astype(int)
print(f"\nClassification report at 0.5 threshold:")
print(classification_report(y_val, preds_binary))

# At optimal threshold (maximizes F1)
precisions, recalls, thresholds = precision_recall_curve(y_val, val_preds_full)
f1_scores = 2 * precisions * recalls / (precisions + recalls + 1e-8)
best_threshold = thresholds[f1_scores[:-1].argmax()]
preds_optimal = (val_preds_full >= best_threshold).astype(int)
print(f"Optimal threshold: {best_threshold:.4f}")
print(f"\nClassification report at optimal threshold:")
print(classification_report(y_val, preds_optimal))

ROC-AUC:  0.9481
PR-AUC:   0.4287

Classification report at 0.5 threshold:
              precision    recall  f1-score   support

           0       0.97      0.96      0.97     75930
           1       0.38      0.42      0.40      4012

    accuracy                           0.94     79942
   macro avg       0.68      0.69      0.68     79942
weighted avg       0.94      0.94      0.94     79942

Optimal threshold: 0.0061

Classification report at optimal threshold:
              precision    recall  f1-score   support

           0       1.00      0.89      0.94     75930
           1       0.32      0.96      0.48      4012

    accuracy                           0.89     79942
   macro avg       0.66      0.92      0.71     79942
weighted avg       0.96      0.89      0.92     79942



In [5]:
# Split val into calibration (50%) and test (50%)
X_cal, X_test, y_cal, y_test = train_test_split(
    X_val, y_val,
    test_size=0.5,
    random_state=42,
    stratify=y_val
)

print(f"Calibration set: {X_cal.shape}, churn rate: {y_cal.mean():.4f}")
print(f"Test set:        {X_test.shape}, churn rate: {y_test.mean():.4f}")

# Get raw predictions on calibration set
cal_raw_preds = final_model.predict_proba(X_cal)[:, 1]

# Fit isotonic regression on calibration set
iso_reg = IsotonicRegression(out_of_bounds='clip')
iso_reg.fit(cal_raw_preds, y_cal)

# Apply calibration to test set
test_raw_preds = final_model.predict_proba(X_test)[:, 1]
test_cal_preds = iso_reg.predict(test_raw_preds)

# Compare metrics on test set
roc_raw = roc_auc_score(y_test, test_raw_preds)
roc_cal = roc_auc_score(y_test, test_cal_preds)
pr_raw  = average_precision_score(y_test, test_raw_preds)
pr_cal  = average_precision_score(y_test, test_cal_preds)

print(f"\n{'Metric':<12} {'Raw':<10} {'Calibrated'}")
print("-" * 35)
print(f"{'ROC-AUC':<12} {roc_raw:.4f}     {roc_cal:.4f}")
print(f"{'PR-AUC':<12} {pr_raw:.4f}     {pr_cal:.4f}")

# Optimal threshold
precisions, recalls, thresholds = precision_recall_curve(y_test, test_cal_preds)
f1_scores = 2 * precisions * recalls / (precisions + recalls + 1e-8)
best_threshold_cal = thresholds[f1_scores[:-1].argmax()]
preds_cal_optimal  = (test_cal_preds >= best_threshold_cal).astype(int)

print(f"\nCalibrated at optimal threshold ({best_threshold_cal:.4f}):")
print(classification_report(y_test, preds_cal_optimal))

# Save
joblib.dump(iso_reg, Path.home() / "kkbox-churn" / "models" / "isotonic_calibrator.pkl")
joblib.dump(best_threshold_cal, Path.home() / "kkbox-churn" / "models" / "optimal_threshold.pkl")
print("Saved.")

Calibration set: (39971, 101), churn rate: 0.0502
Test set:        (39971, 101), churn rate: 0.0502

Metric       Raw        Calibrated
-----------------------------------
ROC-AUC      0.9482     0.9498
PR-AUC       0.4336     0.4136

Calibrated at optimal threshold (0.2500):
              precision    recall  f1-score   support

           0       1.00      0.89      0.94     37965
           1       0.32      0.95      0.48      2006

    accuracy                           0.89     39971
   macro avg       0.66      0.92      0.71     39971
weighted avg       0.96      0.89      0.92     39971

Saved.


In [6]:
feat_imp = pd.DataFrame({
    'feature': X_train.columns,
    'importance': final_model.feature_importances_
}).sort_values('importance', ascending=False)

print(feat_imp.head(20).to_string())

                                feature  importance
4           days_since_last_transaction        1401
7                        days_to_expiry        1131
35                                 city         284
3                current_payment_method         238
92  days_between_last_listen_and_expiry         223
33                            t1_amount         194
5                days_since_last_cancel         152
31                    avg_interval_days         141
99                auto_not_cancel_ratio         130
41                     account_age_days         128
6                  customer_tenure_days         116
23                      autorenew_last3         107
63                    active_days_trend          89
37                             bd_clean          86
69                       final_day_secs          85
1                     current_is_cancel          84
62                listening_trend_ratio          80
14                          pay_per_day          78
13          

In [7]:
# Feature importance
feat_imp = pd.DataFrame({
    'feature': final_model.feature_name_,
    'split_importance': final_model.feature_importances_
}).sort_values('split_importance', ascending=False)

# SHAP importance
X_shap = X_val.sample(3000, random_state=42)
explainer = shap.TreeExplainer(final_model)
shap_values = explainer.shap_values(X_shap)

if isinstance(shap_values, list):
    shap_vals = shap_values[1]
else:
    shap_vals = shap_values

shap_imp = pd.DataFrame({
    'feature': X_shap.columns,
    'shap_importance': np.abs(shap_vals).mean(0)
})

# Merge both
combined = feat_imp.merge(shap_imp, on='feature')

# Features where BOTH are zero
both_zero = combined[
    (combined['split_importance'] == 0) & 
    (combined['shap_importance'] == 0.0)
]

print(f"Total features: {len(combined)}")
print(f"\nFeatures where BOTH split importance AND SHAP = 0: {len(both_zero)}")
print(both_zero.to_string())

Total features: 101

Features where BOTH split importance AND SHAP = 0: 10
                     feature  split_importance  shap_importance
91         switched_off_flag                 0              0.0
92     has_listening_history                 0              0.0
93         stopped_listening                 0              0.0
94         has_listening_15d                 0              0.0
95           is_extreme_risk                 0              0.0
96           is_silent_loyal                 0              0.0
97         auto_off_inactive                 0              0.0
98         has_listening_30d                 0              0.0
99   is_invalid_registration                 0              0.0
100            discount_flag                 0              0.0


/opt/anaconda3/envs/kkbox/lib/python3.11/site-packages/shap/explainers/_tree.py:620: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(


In [8]:
zero_features = [
    'switched_off_flag', 'has_listening_history', 'stopped_listening',
    'has_listening_15d', 'is_extreme_risk', 'is_silent_loyal',
    'auto_off_inactive', 'has_listening_30d', 'is_invalid_registration',
    'discount_flag'
]

# Full model (already trained above, don't reload)
full_auc = roc_auc_score(y_val, final_model.predict_proba(X_val)[:, 1])

# Retrain without zero features (no save)
X_train_reduced = X_train.drop(columns=zero_features)
X_val_reduced   = X_val.drop(columns=zero_features)

reduced_model = lgb.LGBMClassifier(**best_params)
reduced_model.fit(
    X_train_reduced, y_train,
    eval_set=[(X_val_reduced, y_val)],
    callbacks=[
        lgb.early_stopping(stopping_rounds=50, verbose=False),
        lgb.log_evaluation(period=999)
    ]
)

reduced_auc = roc_auc_score(y_val, reduced_model.predict_proba(X_val_reduced)[:, 1])

print("── Ablation: Zero-Importance Feature Removal ──")
print(f"Full model    (101 features): ROC-AUC = {full_auc:.4f}")
print(f"Reduced model  (91 features): ROC-AUC = {reduced_auc:.4f}")
print(f"AUC drop: {full_auc - reduced_auc:.4f}")
print(f"\nConclusion: {len(zero_features)} features with zero split + SHAP importance")
print(f"still contribute {full_auc - reduced_auc:.4f} AUC through interaction effects.")
print(f"Kept all 101 features based on empirical validation.")

── Ablation: Zero-Importance Feature Removal ──
Full model    (101 features): ROC-AUC = 0.9481
Reduced model  (91 features): ROC-AUC = 0.9455
AUC drop: 0.0025

Conclusion: 10 features with zero split + SHAP importance
still contribute 0.0025 AUC through interaction effects.
Kept all 101 features based on empirical validation.


In [9]:
DATA_DIR = Path.home() / "kkbox-churn/data/parquet"
MODEL_DIR = Path.home() / "kkbox-churn/models"

# Load master dataset
df = pd.read_parquet(DATA_DIR / "master_dataset.parquet")

# Load model to get exact feature names
model = joblib.load(MODEL_DIR / "lgbm_churn_final.pkl")
feature_cols = model.feature_name_

# Build feature store: msno + 101 features only
feature_store = df[['msno'] + feature_cols].copy()

# Set msno as index for fast lookup
feature_store = feature_store.set_index('msno')

# Save
feature_store.to_parquet(MODEL_DIR / "kkbox_feature_store.parquet")
print(f"Feature store saved: {feature_store.shape}")
print(f"Size: {(MODEL_DIR / 'kkbox_feature_store.parquet').stat().st_size / 1024 / 1024:.1f} MB")

Feature store saved: (970960, 101)
Size: 198.7 MB
